In [22]:
import numpy as np
import pandas as pd
from bertopic import BERTopic
from octis.evaluation_metrics.diversity_metrics import InvertedRBO, TopicDiversity
from bertopic.representation import MaximalMarginalRelevance
from spacy.lang.en.stop_words import STOP_WORDS
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from bertopic.representation import KeyBERTInspired
from transformers import AutoTokenizer



In [23]:
data = np.load('/home/banfi/TETYS/pipeline/src/python/data/interim/embeddings/science_news/science_news_embeddings_aggregation_with_lemma.npz',allow_pickle=True) 


texts = data['text'] 
embeddings = data['embedding'] 
documents = data['clean_text']

df = pd.DataFrame({'full_text':documents})

#PATHS_DATA = {'sn_model':'/home/banfi/TETYS/pipeline/src/python/data/raw/science_news_pipeline_data.parquet'}

#Science news
#PATHS_MODELS = {'sn_model':'/home/banfi/TETYS/pipeline/src/python/models/science_news/model_0.360.safetensors'}
# Scopus
PATHS_MODELS = {'scopus':'/home/banfi/TETYS/pipeline/src/python/models/science_news/model_0.309.safetensors'}

EMBEDDING_MODEL = 'codefuse-ai/F2LLM-0.6B'

for group in PATHS_MODELS.keys():
    #log.info(f"### ### ### ### Evaluating model for {group}... ### ### ### ###")
    path_model = PATHS_MODELS[group]
    model = BERTopic.load(path_model, embedding_model=EMBEDDING_MODEL)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 623.09it/s, Materializing param=norm.weight]                              


In [24]:
topics = model.topics_

model.update_topics(documents,
topics=topics,
vectorizer_model=CountVectorizer( stop_words=list(STOP_WORDS),
token_pattern=r"(?u)\b\w+\b", ngram_range=(1, 1) ),
ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=False),
representation_model=None
)

2026-01-28 18:30:41,550 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.


In [26]:
model.get_topics()

{-1: [('study', 0.016680940450649037),
  ('virus', 0.015424254073991206),
  ('new', 0.014994750682021175),
  ('bacteria', 0.01489223142252776),
  ('disease', 0.014577944274355261),
  ('infection', 0.01447060394551537),
  ('treatment', 0.013820061848099328),
  ('suggest', 0.013719784529902369),
  ('cancer', 0.01369279370782253),
  ('use', 0.013584380366379867)],
 0: [('vaccine', 0.07441310650000578),
  ('vaccination', 0.03170097609794195),
  ('s', 0.024686566438445516),
  ('u', 0.02463979812174142),
  ('pfizer', 0.021752651111211076),
  ('shot', 0.021692648579619916),
  ('effectiveness', 0.020873892388160335),
  ('moderna', 0.018987557371685435),
  ('mrna', 0.01889778434830626),
  ('child', 0.01866646207530385)],
 1: [('hiv', 0.09755407856879465),
  ('drug', 0.02902490896864797),
  ('treatment', 0.025693038187172126),
  ('cell', 0.025390946176299375),
  ('study', 0.02514309171258288),
  ('infect', 0.02228048460096824),
  ('immune', 0.020757308762814165),
  ('reduce', 0.02024094144462534

In [17]:
keywords_sample = { 'topics' : [ ['a','b','c','d','e','f','g','h','i','l'],['a','b','c','d','e','f','g','h','i','l'],['a','b','c','d','e','f','g','h','i','l'] ] }

In [18]:
sublist = ['a','b','c','d','e','f','g','h','i','l']

In [14]:
reverse_sublist = sublist[::-1]

In [16]:
keywords_sample = { 'topics' : [ sublist, reverse_sublist ] }

In [25]:
    # Evaluate Topic Diversity
    model_output = {}
    model_output["topics"] = model.get_topics()
    model_output["topics"] = {
        k: v for k, v in model_output["topics"].items() if k != -1
    }
    model_output["topics"] = [
        [x[0] for x in l] for l in model_output["topics"].values()
    ]


    topic_diversity = TopicDiversity()
    divesity = topic_diversity.score(model_output)
    print(f"Topic Diversity: {divesity}")


    ## Compute InvertedRBO
    inverted_rbo = InvertedRBO()
    inverted_rbo_score = inverted_rbo.score(model_output)
    print(f"Inverted RBO: {inverted_rbo_score}")


Topic Diversity: 0.7379310344827587
Inverted RBO: 0.9747765631871217
